In [ ]:
# Figure plotting: EVI encroachment–retreat hexbin plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, ScalarFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ======== USER SETTINGS ========
CSV_PATH = "/home/jovyan/DEA products paper/Results/step 5_evi_v2/tables_v3/evi_plausibility_batch_1988_2023_5yr_plusEpochs_20251014_005533.csv"
SAVE_PATH = "Figure_7_hexbin_log_final.png"

CORE_LIM = 0.20
THRESH_NOCHANGE = 0.02
GUIDE = 0.05

GRIDSIZE = 55
DPI = 300
# ===============================

# ---- Load data ----
df = pd.read_csv(CSV_PATH)

required_cols = {"SubRegion", "Period", "ΔEVI_med_enc", "ΔEVI_med_ret"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV missing columns: {missing}")

df = df.dropna(subset=["ΔEVI_med_enc", "ΔEVI_med_ret"]).copy()

# ---- Filter to plotting extent ----
mask = (
    (df["ΔEVI_med_enc"].abs() <= CORE_LIM) &
    (df["ΔEVI_med_ret"].abs() <= CORE_LIM)
)

x = df.loc[mask, "ΔEVI_med_enc"]
y = df.loc[mask, "ΔEVI_med_ret"]

# ---- Create figure ----
fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ---- No-change zone (±0.02) ----
ax.fill_betweenx(
    [-CORE_LIM, CORE_LIM],
    -THRESH_NOCHANGE, THRESH_NOCHANGE,
    color="0.85", alpha=0.50, zorder=0
)
ax.fill_between(
    [-CORE_LIM, CORE_LIM],
    -THRESH_NOCHANGE, THRESH_NOCHANGE,
    color="0.85", alpha=0.50, zorder=0
)

# ---- Hexbin with log scaling ----
hb = ax.hexbin(
    x, y,
    gridsize=GRIDSIZE,
    extent=(-CORE_LIM, CORE_LIM, -CORE_LIM, CORE_LIM),
    mincnt=1,
    cmap="viridis",
    bins="log",
    linewidths=0,
    zorder=2
)

# ---- Reference axes ----
ax.axhline(0, color="0.35", lw=0.9, zorder=3)
ax.axvline(0, color="0.35", lw=0.9, zorder=3)

# ---- 1:1 diagonal ----
ax.plot(
    [-CORE_LIM, CORE_LIM],
    [-CORE_LIM, CORE_LIM],
    ls="--", lw=1.2, color="0.2", alpha=0.8, zorder=3
)

# ---- ±0.05 guide lines ----
ax.axhline(+GUIDE, color="0.6", lw=0.9, ls="--", zorder=1)
ax.axhline(-GUIDE, color="0.6", lw=0.9, ls="--", zorder=1)
ax.axvline(+GUIDE, color="0.6", lw=0.9, ls="--", zorder=1)
ax.axvline(-GUIDE, color="0.6", lw=0.9, ls="--", zorder=1)

# ---- Labels and limits ----
ax.set_xlim(-CORE_LIM, CORE_LIM)
ax.set_ylim(-CORE_LIM, CORE_LIM)
ax.set_xlabel("Median ΔEVI (Encroachment)", fontsize=12)
ax.set_ylabel("Median ΔEVI (Retreat)", fontsize=12)
ax.set_title("")

# ---- Ensure square plot ----
ax.set_aspect("equal", adjustable="box")

# ---- Grid ----
ax.grid(True, linestyle=":", linewidth=0.4, alpha=0.20)

# ---- PERFECT COLORBAR ALIGNMENT ----
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.20)

cbar = fig.colorbar(hb, cax=cax)
cbar.set_label("Number of wetland–period observations\n(log scale)", fontsize=10, labelpad=10)

# Better ticks
cbar.locator = LogLocator(base=10, subs=[1, 3])
cbar.formatter = ScalarFormatter()
cbar.update_ticks()

# ---- Save ----
fig.savefig(
    SAVE_PATH,
    dpi=DPI,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
print(f"Saved figure: {SAVE_PATH}")